In [60]:
import numpy as np
from pprint import pprint

In [61]:
#try to take as input a npz file, output a pdb file and then compute the force field using openmm

directory_npz = '/data2/scratch/group_scratch/boltz_train/rcsb_processed_targets/structures'
filename_npz = '2ags.npz'

In [62]:
import numpy as np

data = np.load(f'{directory_npz}/{filename_npz}', allow_pickle=True)

In [63]:
from pathlib import Path
from boltz.data.types import Structure 
from boltz.data.write.pdb import to_pdb

# Prima carichiamo la struttura
input_file = Path(f'{directory_npz}/{filename_npz}')
structure = Structure.load(input_file)

# Poi ispezionaimo i dettagli della struttura
print("Chiavi disponibili nella struttura:")
print(structure.__dict__.keys())

print("\nPrimo residuo:")
if len(structure.residues) > 0:
    pprint(structure.residues[0])

print("\nPrimo atomo:")
if len(structure.atoms) > 0:
    pprint(structure.atoms[0])

print("\nNumero di catene:", len(structure.chains))
print("\nPrima catena:")
pprint(structure.chains[0])

# Definizione della funzione di conversione
def simplified_to_pdb(structure):
    """Versione semplificata della conversione in PDB"""
    pdb_lines = []
    atom_index = 1
    
    for chain_idx, chain in enumerate(structure.chains):
        chain_tag = chr(65 + chain_idx)  # Usa lettere A, B, C, ecc. per le catene
        
        res_start = chain["res_idx"]
        res_end = chain["res_idx"] + chain["res_num"]
        
        residues = structure.residues[res_start:res_end]
        for residue in residues:
            res_name = str(residue[0])  # Nome del residuo
            atom_start = residue[2]  # Indice di inizio degli atomi
            atom_end = atom_start + residue[4]  # Numero di atomi
            
            atoms = structure.atoms[atom_start:atom_end]
            for i, atom in enumerate(atoms):
                if not atom[5]:  # Se l'atomo non è presente
                    continue
                    
                # Usa ATOM come default per tutti gli atomi
                record_type = "ATOM"
                name = f" {atom[0][0]:>3}"  # Nome dell'atomo
                alt_loc = " "
                res_num = residue[1]
                pos = atom[4]  # Coordinate
                occupancy = 1.00
                temp_factor = 0.00
                element = " C"  # Default a carbonio
                charge = "  "
                
                # Formato PDB standard
                atom_line = (
                    f"{record_type:<6}{atom_index:>5} {name:<4}{alt_loc:>1}"
                    f"{res_name:>3} {chain_tag:>1}"
                    f"{res_num:>4}    "
                    f"{pos[0]:>8.3f}{pos[1]:>8.3f}{pos[2]:>8.3f}"
                    f"{occupancy:>6.2f}{temp_factor:>6.2f}          "
                    f"{element}{charge}"
                )
                pdb_lines.append(atom_line)
                atom_index += 1
                
        # Aggiungi TER alla fine di ogni catena
        ter_line = f"TER   {atom_index:>5}      {res_name:>3} {chain_tag:>1}{res_num:>4}"
        pdb_lines.append(ter_line)
        atom_index += 1
    
    pdb_lines.append("END")
    return "\n".join(pdb_lines)

# Converti la struttura in PDB
pdb_content = simplified_to_pdb(structure)

Chiavi disponibili nella struttura:
dict_keys(['atoms', 'bonds', 'residues', 'chains', 'connections', 'interfaces', 'mask'])

Primo residuo:
('MET', 14, 0, 0, 8, 1, 4, True, False)

Primo atomo:
([46,  0,  0,  0], 7, 0, [0., 0., 0.], [ 1.8903918 , -1.5252995 , -0.42638594], False, 0)

Numero di catene: 4

Prima catena:
('A1', 0, 0, 0, 0, 0, 5013, 0, 652)


In [64]:
# Ispeziona la struttura per vedere se ci sono catene valide


In [66]:
output_file = 'prova.pdb'
with open(output_file, 'w') as f:
    f.write(pdb_content)
